# Colab runner

Runs this repo on a Colab GPU. The code lives in git, not in this notebook.

**With the VS Code Colab extension**, this notebook's kernel runs on the Colab VM, but your
local files do *not* exist there. Workflow: edit locally → `git push` your branch → re-run the
**Get the code** cell (it pulls) → run.

Set `REPO_URL` and `BRANCH` below. Put `runs/` and results on Google Drive so they survive
the VM being recycled.

In [ ]:
REPO_URL = "https://github.com/zianye183/ECE661-LeJEPA.git"
BRANCH = "main"                                     # or your feature branch
WORKDIR = "/content/ece661"
DRIVE_DIR = "/content/drive/MyDrive/ece661"         # shared folder for runs + results

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Google Drive (optional but recommended)
If mounting fails under the VS Code extension, skip it and set `DRIVE_DIR = WORKDIR`; then
download `runs/` and `results/` before the runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Get the code (re-run after every `git push`)

In [ ]:
import os
if not os.path.exists(WORKDIR):
    !git clone -b {BRANCH} {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} fetch origin && git -C {WORKDIR} checkout {BRANCH} && git -C {WORKDIR} pull --ff-only
%cd {WORKDIR}
!git log -1 --oneline
!pip install -q -e ".[dev]"

## Quick checks (seconds)

In [ ]:
!python -m pytest -q tests/test_sigreg.py tests/test_losses.py tests/test_eval.py

## Cost-per-run probe (G2 gate)
One epoch on CIFAR-10 for each method. `metrics.jsonl` records `time_s` and `peak_mem_gb`;
multiply by the epoch count to size the budget.

In [ ]:
for m in ["lejepa", "simclr", "simsiam"]:
    !python train.py --method {m} --set experiment=cost_probe --set train.epochs=1 --skip-eval --runs-dir {DRIVE_DIR}/runs

## A real run

In [ ]:
METHOD, DATASET, SEED = "lejepa", "cifar10", 0
!python train.py --method {METHOD} --seed {SEED} --set data.dataset={DATASET} --set experiment=exp1_repro \
    --runs-dir {DRIVE_DIR}/runs --results-dir {DRIVE_DIR}/results --resume

In [ ]:
!python scripts/collect_results.py {DRIVE_DIR}/results